# Obtaining Data from AWS for Classification Task

# Data Download for Classification Task

We need the data to complete the classification challenge task.  

We **could** download the files and upload them to the repo, but that approach is **not repeatable**.  

Instead, we create a script that downloads the data directly from its source. The resulting data files **are not committed**, so the process remains reproducible.  

The script below uses the credentials provided with the challenge to connect to the AWS S3 bucket via a `boto3` session, and downloads the files into the `data/` folder.  

If the folder doesn’t exist yet, the script will create it automatically.  

We will work through the analysis **cell by cell**, with comments explaining each step.

In [1]:
# Imports 

import os
import json
import requests
import boto3

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
# Specify the credentials location -- location taken from the Challenge Readme, from the penultimate paragraph of the 
# section: 0. Setup: Data.

creds_url = "https://cct-ds-code-challenge-input-data.s3.af-south-1.amazonaws.com/ds_code_challenge_creds.json"

In [3]:
# Obtain credentials in session. 

creds_response = requests.get(creds_url)
creds = creds_response.json()

In [4]:
# Assign access and secret key objects. 

AWS_ACCESS_KEY = creds['s3']['access_key']
AWS_SECRET_KEY = creds['s3']['secret_key']

In [5]:
# Create a boto3 session using Credentials and the Region Name given to us in the Challenge ReadMe. 

# Create Session
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="af-south-1"
)

# Create S3 Client 
s3 = session.client("s3")

In [6]:
# Create a data dictionary (if it exists this request is ignored)

# os.makedirs("~/data", exist_ok=True)
data_dir = "data"          # ./data
os.makedirs(data_dir, exist_ok=True)

In [7]:
# Specify a the S3 bucket and file name of the data we want to obtain. 

bucket = "cct-ds-code-challenge-input-data"
file_name = "city-hex-polygons-8.geojson"

# specify local path
local_path = f"data/{file_name}"

# download the bucket
s3.download_file(bucket, file_name, local_path)

# confirm download. 
print(f"Downloaded to {local_path}")

Downloaded to data/city-hex-polygons-8.geojson


In [9]:
# bucket and file name of the data we want to obtain. 

bucket = "cct-ds-code-challenge-input-data"
file_name = "sr_hex.csv.gz"

# specify local path
local_path = f"data/{file_name}"

# download the bucket
s3.download_file(bucket, file_name, local_path)

# confirm download. 
print(f"Downloaded to {local_path}")

Downloaded to data/sr_hex.csv.gz


In [10]:
import random
# S3 setup
bucket = "cct-ds-code-challenge-input-data"

# Prefixes for classes
prefixes = {"yes": "images/swimming-pool/yes", "no": "images/swimming-pool/no"}

# Local folders
local_root = "data/sample_images"
os.makedirs(local_root, exist_ok=True)

# Number of images to sample per class
n_samples = 100

for label, prefix in prefixes.items():
    local_folder = os.path.join(local_root, label)
    os.makedirs(local_folder, exist_ok=True)
    
    # List objects in S3
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    all_keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".tif")]
    
    # Sample
    sampled_keys = random.sample(all_keys, min(n_samples, len(all_keys)))
    
    # Download
    for key in sampled_keys:
        local_path = os.path.join(local_folder, os.path.basename(key))
        s3.download_file(bucket, key, local_path)
        print(f"Downloaded {key} -> {local_path}")

Downloaded images/swimming-pool/yes/W17B_6_2.tif -> data/sample_images/yes/W17B_6_2.tif
Downloaded images/swimming-pool/yes/W16C_23_48.tif -> data/sample_images/yes/W16C_23_48.tif
Downloaded images/swimming-pool/yes/W17B_5_43.tif -> data/sample_images/yes/W17B_5_43.tif
Downloaded images/swimming-pool/yes/W17A_9_13.tif -> data/sample_images/yes/W17A_9_13.tif
Downloaded images/swimming-pool/yes/W17B_18_79.tif -> data/sample_images/yes/W17B_18_79.tif
Downloaded images/swimming-pool/yes/W17B_11_46.tif -> data/sample_images/yes/W17B_11_46.tif
Downloaded images/swimming-pool/yes/W17B_22_31.tif -> data/sample_images/yes/W17B_22_31.tif
Downloaded images/swimming-pool/yes/W17C_5_67.tif -> data/sample_images/yes/W17C_5_67.tif
Downloaded images/swimming-pool/yes/W17A_10_57.tif -> data/sample_images/yes/W17A_10_57.tif
Downloaded images/swimming-pool/yes/W17C_4_29.tif -> data/sample_images/yes/W17C_4_29.tif
Downloaded images/swimming-pool/yes/W17B_2_87.tif -> data/sample_images/yes/W17B_2_87.tif
Do